In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================
import os

# DeepFace / TensorFlow never loads in this kernel - it runs in a separate CPU worker (cell 2)

import json, shutil, time
from pathlib import Path
import numpy as np, cv2, torch
from PIL import Image, ImageEnhance

torch.backends.cudnn.enabled = False   # avoids CUDNN_STATUS_SUBLIBRARY_LOADING_FAILED on this machine
from IPython.display import display

# ---------------- folders ----------------
PROJECT_DIR = Path.cwd()
USERS_DIR   = PROJECT_DIR / "users"
AVATAR_DIR  = PROJECT_DIR / "Avatar"
CROP_DIR    = PROJECT_DIR / "crop"       # face crops used by the model
OUTPUT_DIR  = PROJECT_DIR / "output"     # final avatars
REJECT_DIR  = PROJECT_DIR / "rejected"   # blocked images + reason
for d in (CROP_DIR, OUTPUT_DIR, REJECT_DIR):
    d.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# ---------------- avatars ----------------
AVATAR_PATHS = {
    "Man":         AVATAR_DIR / "Saytara_male.jpg",
    "Woman":       AVATAR_DIR / ("Saytara_Femal.png"),        # cleaned template: no ghost streaks
    "Woman_Hijab": AVATAR_DIR / "Saytara_hijab.jpg",
}

# ---------------- face rules ----------------
FACE_DETECTOR       = "retinaface"
MIN_FACE_CONFIDENCE = 0.90   # ignore weak detections
MIN_FACE_SIZE_PX    = 80     # ignore tiny background faces
MAIN_FACE_DOMINANCE = 2.5    # with several faces, the biggest must be >= 2.5x the 2nd, else reject
CROP_SCALE          = 2.0    # crop around the face (includes hair / hijab)

# ---------------- classifiers ----------------
SCOPE_MIN_PERSON_SCORE     = 0.35   # 'person' must be the top label; this is only a floor
MIN_GENDER_CONFIDENCE      = 75.0   # DeepFace %, below this -> CLIP fallback
MIN_CLIP_GENDER_CONFIDENCE = 0.75   # CLIP fallback, below this -> reject
GLASSES_POSITIVE_THRESHOLD = 0.58
GLASSES_MIN_SCORE_MARGIN   = 0.08
BEARD_POSITIVE_THRESHOLD   = 0.56
BEARD_MIN_SCORE_MARGIN     = 0.05
HIJAB_POSITIVE_THRESHOLD   = 0.50
HIJAB_MIN_SCORE_MARGIN     = 0.00
HIJAB_PROMPT_PAIRS = [
    ("a photo of a woman wearing a hijab that covers her hair and neck",
     "a photo of a woman with visible uncovered hair around her face"),
    ("a portrait of a woman in a headscarf wrapped around her head",
     "a portrait of a woman with her natural hair visible around her head"),
    ("a woman wearing a cloth scarf that frames her face and covers her hair",
     "a woman with an uncovered hairstyle visible on top and at the sides"),
]

# Manual fixes for known images (key = file name without extension)
OVERRIDES = {
    "1790583585959": {"gender": "Woman"},
    "1790261466309": {"hijab": False},
}

# ---------------- models ----------------
CLIP_DIR   = Path("/Models/clip-vit-base-patch32")
CLIP_MODEL = str(CLIP_DIR) if CLIP_DIR.exists() else "openai/clip-vit-base-patch32"
FLUX_MODEL = "black-forest-labs/FLUX.2-klein-9B"
LORA       = "sisniha/BFS-Best-Face-Swap"
LORA_FILE  = "bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors"

# ---------------- generation ----------------
STEPS              = 20     # proven settings (male test image)
CFG = 3.0  # 2.5 gave grainy skin and scratchy hair/beard
REFINE_CFG = 2.0  # identity refine pass: softer, keeps detail without grain
LORA_STRENGTH      = 1.1
SEED               = 42
HEAD_CROP_SCALE    = 1.6    # crop sent to FLUX: face + hair, face large and clear
REFINE_IDENTITY    = True   # 2nd pass: re-apply the face from a tight close-up (better likeness)
REFINE_STRENGTH    = 0.85
FACE_CROP_SCALE    = 0.75   # tight close-up for the refine pass

CLEAN_BG           = True   # remove ghost streaks: only emblem + person kept, rest pure black

# ---------------- glasses validation ----------------
VALIDATE_GLASSES   = True   # check the avatar visor survived, repair it if not
VISOR_MIN_SCORE    = 0.38   # tested: changed glasses = 0.26, correct glasses = 0.45-0.49
DARK_FRAME_RATIO   = 1.3    # more black frame lines than the avatar = ordinary eyeglasses
MAX_GLASSES_FIXES  = 3
ADD_AVATAR_GLASSES = True       # every avatar gets the blue futuristic glasses
USER_GLASSES_MODE  = "avatar"   # user wears glasses: "avatar" = replace with avatar glasses, "keep" = keep theirs

NOTEBOOK_VERSION = "v2: crop reference + preview reuse + beard validation + composition prompt"
print("Notebook version:", NOTEBOOK_VERSION)
print("Config OK | CLIP:", CLIP_MODEL)
for k, p in AVATAR_PATHS.items():
    print(f"  {k:12s} -> {p}  {'OK' if p.exists() else 'MISSING'}")


In [ ]:
# ============================================================
# 2. LOAD MODELS (once)
#    DeepFace (TensorFlow) is NOT imported here - TF + PyTorch in one process
#    crashes the kernel on this machine. DeepFace runs in a separate CPU worker.
# ============================================================
import os, sys, json, subprocess, textwrap, warnings
from pathlib import Path
import numpy as np, cv2, torch
warnings.filterwarnings("ignore")

_missing = [n for n in ("PROJECT_DIR", "OUTPUT_DIR", "FACE_DETECTOR", "CROP_SCALE",
                        "CLIP_MODEL", "FLUX_MODEL", "LORA", "LORA_FILE") if n not in globals()]
if _missing:
    raise RuntimeError(f"Run cell 1 (CONFIG) first - missing: {', '.join(_missing)}")

assert torch.cuda.is_available(), "PyTorch cannot see the GPU - FLUX needs CUDA."
torch.backends.cudnn.enabled = False          # avoids the cuDNN sub-library error

# ---------------- DeepFace worker (separate process, CPU only) ----------------
# RetinaFace needs Keras 2: make sure tf-keras is installed (one-time)
import importlib.util
if importlib.util.find_spec("tf_keras") is None:
    print("Installing tf-keras (needed by RetinaFace) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tf-keras"], check=True)

if importlib.util.find_spec("rembg") is None:
    print("Installing rembg (clean black background) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rembg", "onnxruntime"], check=True)

WORKER_PATH = PROJECT_DIR / "deepface_worker.py"
WORKER_PATH.write_text(textwrap.dedent("""
    import os, sys, json, io, contextlib, warnings
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"      # TensorFlow on CPU only
    os.environ["TF_USE_LEGACY_KERAS"] = "1"        # RetinaFace needs Keras 2 (pip install tf-keras)
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
    warnings.filterwarnings("ignore")
    import cv2
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        from deepface import DeepFace

    path, detector, crop_scale = sys.argv[1], sys.argv[2], float(sys.argv[3])
    img = cv2.imread(path)
    out = {"faces": []}
    if img is None:
        out["error"] = "unreadable"
    else:
        try:
            faces = DeepFace.extract_faces(img_path=img, detector_backend=detector,
                                           enforce_detection=True, align=False)
        except ValueError:
            faces = []
        H, W = img.shape[:2]
        for f in faces:
            a = f["facial_area"]
            x, y, w, h = int(a["x"]), int(a["y"]), int(a["w"]), int(a["h"])
            cx, cy, nw, nh = x + w // 2, y + h // 2, int(w * crop_scale), int(h * crop_scale)
            crop = img[max(0, cy - nh // 2):min(H, cy + nh // 2), max(0, cx - nw // 2):min(W, cx + nw // 2)]
            gender = {}
            if crop.size:
                with contextlib.redirect_stdout(io.StringIO()):
                    g = DeepFace.analyze(img_path=crop, actions=["gender"], detector_backend="skip",
                                         enforce_detection=False, silent=True)
                g = g[0] if isinstance(g, list) else g
                gender = {k: float(v) for k, v in g["gender"].items()}
            out["faces"].append({"facial_area": {"x": x, "y": y, "w": w, "h": h},
                                 "confidence": float(f.get("confidence", 0)), "gender": gender})
    print("__JSON__" + json.dumps(out))
"""))

def run_deepface_worker(image_path, timeout=600):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": "-1", "TF_CPP_MIN_LOG_LEVEL": "3",
           "TF_USE_LEGACY_KERAS": "1"}
    r = subprocess.run([sys.executable, str(WORKER_PATH), str(image_path), FACE_DETECTOR, str(CROP_SCALE)],
                       capture_output=True, text=True, timeout=timeout, env=env)
    lines = [l for l in r.stdout.splitlines() if l.startswith("__JSON__")]
    if not lines:
        raise RuntimeError("DeepFace worker failed:\n" + (r.stderr or r.stdout)[-1500:])
    return json.loads(lines[-1][len("__JSON__"):])

# test the worker once (first run downloads RetinaFace + gender weights)
if "deepface_ready" not in globals():
    print("Testing DeepFace worker (first run downloads weights) ...")
    _probe = OUTPUT_DIR / "_probe.jpg"
    cv2.imwrite(str(_probe), np.zeros((256, 256, 3), np.uint8))
    print("  worker OK:", run_deepface_worker(_probe))
    _probe.unlink(missing_ok=True)
    deepface_ready = True

# ---------------- CLIP (PyTorch, GPU) ----------------
from transformers import pipeline as hf_pipeline
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
# ---------------- GPU memory check: full GPU or CPU-offload mode ----------------
free_gb  = torch.cuda.mem_get_info()[0] / 1e9
total_gb = torch.cuda.mem_get_info()[1] / 1e9
LOW_VRAM = free_gb < 45
print(f"GPU free: {free_gb:.0f} / {total_gb:.0f} GB -> {'CPU-offload mode (slower)' if LOW_VRAM else 'full GPU mode'}")
if total_gb - free_gb > 5 and "pipe" not in globals():
    print("  NOTE: another process is using the GPU - check `!nvidia-smi` and stop old kernels for full speed.")

if "clip" not in globals():
    print("Loading CLIP ...")
    clip = hf_pipeline("zero-shot-image-classification", model=CLIP_MODEL, device=-1 if LOW_VRAM else 0)
    print("CLIP loaded.")

# ---------------- FLUX + BFS LoRA (PyTorch, GPU) ----------------
from diffusers import Flux2KleinPipeline, FlowMatchEulerDiscreteScheduler
from diffusers.utils import logging as df_logging
df_logging.set_verbosity_error()
if "pipe" not in globals() or pipe is None:
    print("Loading", FLUX_MODEL, "...")
    if LOW_VRAM:
        pipe = Flux2KleinPipeline.from_pretrained(FLUX_MODEL, dtype=torch.bfloat16)
        pipe.enable_model_cpu_offload()       # peak ~20 GB VRAM
    else:
        pipe = Flux2KleinPipeline.from_pretrained(FLUX_MODEL, dtype=torch.bfloat16)
        pipe.to("cuda")
    pipe.load_lora_weights(LORA, weight_name=LORA_FILE, adapter_name="bfs")
    print("FLUX + BFS LoRA loaded.")

pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config(
    pipe.scheduler.config, shift=1.0, use_dynamic_shifting=False
)

# LoRA strength is passed per call (attention_kwargs={"scale": ...}) - no set_adapters()
# during generation, so "requires_grad ... inference tensor" can never happen.
_LORA = {"scale": 1.0}
def set_lora(weight):
    _LORA["scale"] = float(weight)
def lora_kwargs():
    return {"scale": _LORA["scale"]}

print(f"Models ready | GPU: {torch.cuda.get_device_name(0)} | VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")


In [ ]:
# ============================================================
# 2. LOAD MODELS + PREVIEW DETECTED USER FACES
# DeepFace stays in a CPU subprocess because TensorFlow and
# PyTorch together crash this machine's notebook kernel.
# ============================================================
import os
import sys
import json
import subprocess
import importlib.util
from pathlib import Path

import cv2
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

required = ("PROJECT_DIR", "USERS_DIR", "OUTPUT_DIR", "FACE_DETECTOR",
            "CROP_SCALE", "MIN_FACE_CONFIDENCE", "MIN_FACE_SIZE_PX", "MAIN_FACE_DOMINANCE",
            "IMAGE_EXTENSIONS", "CLIP_MODEL", "FLUX_MODEL", "LORA", "LORA_FILE",
            "GLASSES_POSITIVE_THRESHOLD", "GLASSES_MIN_SCORE_MARGIN",
            "BEARD_POSITIVE_THRESHOLD", "BEARD_MIN_SCORE_MARGIN",
            "HIJAB_POSITIVE_THRESHOLD", "HIJAB_MIN_SCORE_MARGIN", "HIJAB_PROMPT_PAIRS")
missing = [name for name in required if name not in globals()]
assert not missing, f"Run cell 1 (CONFIG) first: {missing}"
assert torch.cuda.is_available(), "FLUX needs a CUDA GPU in this notebook."
torch.backends.cudnn.enabled = False  # required workaround on this machine

# Install only missing support packages.
packages = []
if importlib.util.find_spec("tf_keras") is None:
    packages.append("tf-keras")
if importlib.util.find_spec("rembg") is None:
    packages.append("rembg")
if importlib.util.find_spec("onnxruntime") is None:
    packages.append("onnxruntime")
if packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

# DeepFace runs in its own CPU process; cell 3 calls run_deepface_worker().
DEEPFACE_WORKER_CODE = r"""
import os, sys, json, cv2
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
from deepface import DeepFace

path, detector = sys.argv[1:3]
scale = float(sys.argv[3])
min_conf = float(sys.argv[4])
min_size = int(sys.argv[5])
dominance = float(sys.argv[6])
image = cv2.imread(path)
result = {"faces": [], "background_faces_ignored": 0}
if image is None:
    result["error"] = "unreadable"
else:
    height, width = image.shape[:2]
    detected = DeepFace.extract_faces(img_path=image, detector_backend=detector,
                                       enforce_detection=False, align=False)
    candidates = [
        face for face in detected
        if float(face.get("confidence", 0)) >= min_conf
        and int(face["facial_area"]["w"]) >= min_size
        and int(face["facial_area"]["h"]) >= min_size
    ]
    if candidates:
        candidates.sort(key=lambda face: face["facial_area"]["w"] * face["facial_area"]["h"],
                        reverse=True)
        largest_area = candidates[0]["facial_area"]["w"] * candidates[0]["facial_area"]["h"]
        main_faces = [
            face for face in candidates
            if face["facial_area"]["w"] * face["facial_area"]["h"] >= largest_area / dominance
        ]
        if len(main_faces) > 1:
            result["error"] = "multiple_main_faces"
            result["main_faces"] = len(main_faces)
        else:
            face = main_faces[0]
            area = face["facial_area"]
            x, y, w, h = (int(area[key]) for key in ("x", "y", "w", "h"))
            cx, cy = x + w // 2, y + h // 2
            nw, nh = int(w * scale), int(h * scale)
            crop = image[max(0, cy - nh // 2):min(height, cy + nh // 2),
                         max(0, cx - nw // 2):min(width, cx + nw // 2)]
            if crop.size:
                analysis = DeepFace.analyze(img_path=crop, actions=["gender"],
                                            detector_backend="skip",
                                            enforce_detection=False, silent=True)
                analysis = analysis[0] if isinstance(analysis, list) else analysis
                result["faces"] = [{
                    "facial_area": {"x": x, "y": y, "w": w, "h": h},
                    "confidence": float(face.get("confidence", 0)),
                    "gender": {key: float(value) for key, value in analysis["gender"].items()}
                }]
                result["background_faces_ignored"] = len(candidates) - 1
print("__JSON__" + json.dumps(result))
"""

def run_deepface_worker(image_path, timeout=600):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": "-1",
           "TF_USE_LEGACY_KERAS": "1", "TF_CPP_MIN_LOG_LEVEL": "3"}
    process = subprocess.run(
        [sys.executable, "-c", DEEPFACE_WORKER_CODE, str(image_path),
         FACE_DETECTOR, str(CROP_SCALE), str(MIN_FACE_CONFIDENCE),
         str(MIN_FACE_SIZE_PX), str(MAIN_FACE_DOMINANCE)],
        capture_output=True, text=True, timeout=timeout, env=env
    )
    payload = next((line[len("__JSON__"):] for line in reversed(process.stdout.splitlines())
                    if line.startswith("__JSON__")), None)
    if payload is None:
        raise RuntimeError("DeepFace worker failed:\n" + (process.stderr or process.stdout)[-1500:])
    return json.loads(payload)

# CLIP is reused by cell 3 for scope, glasses, and hijab checks.
from transformers import pipeline as hf_pipeline
free_gb, total_gb = (value / 1e9 for value in torch.cuda.mem_get_info())
LOW_VRAM = free_gb < 45
if "clip" not in globals():
    clip = hf_pipeline("zero-shot-image-classification", model=CLIP_MODEL,
                       device=-1 if LOW_VRAM else 0)

# FLUX and its face-swap LoRA are reused by cell 4.
from diffusers import Flux2KleinPipeline, FlowMatchEulerDiscreteScheduler
if not isinstance(globals().get("pipe"), Flux2KleinPipeline):
    pipe = Flux2KleinPipeline.from_pretrained(FLUX_MODEL, dtype=torch.bfloat16)
    if LOW_VRAM:
        pipe.enable_model_cpu_offload()
    else:
        pipe.to("cuda")
    pipe.load_lora_weights(LORA, weight_name=LORA_FILE, adapter_name="bfs")
pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config(
    pipe.scheduler.config, shift=1.0, use_dynamic_shifting=False
)
print(f"Models ready | GPU: {torch.cuda.get_device_name(0)} | "
      f"free VRAM before loading: {free_gb:.1f}/{total_gb:.1f} GB")

# Preview up to three faces with a wider, editable margin.
PREVIEW_LIMIT = 10
PREVIEW_CROP_SCALE = 2.0  # 1.0 = tight face box; 2.0 adds half a face width on each side
FACE_PREVIEW_RESULTS = {}
GLASSES_PREVIEW_LABELS = ("a close-up portrait of a person wearing eyeglasses",
                         "a close-up portrait of a person without eyeglasses")

def preview_pair_score(image, positive, negative):
    scores = {item["label"]: float(item["score"])
              for item in clip(image, candidate_labels=[positive, negative])}
    return scores[positive] / max(scores[positive] + scores[negative], 1e-12)
preview_files = sorted(p for p in USERS_DIR.iterdir()
                       if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS) if USERS_DIR.exists() else []
if not preview_files:
    print("No user images found in users/ for the face preview.")

shown = 0
for user_path in preview_files:
    if shown >= PREVIEW_LIMIT:
        break
    result = run_deepface_worker(user_path)
    if result.get("error") == "multiple_main_faces":
        print(f"{user_path.name}: BLOCKED — {result['main_faces']} main faces detected")
        continue
    if not result["faces"]:
        print(f"{user_path.name}: no clear face detected")
        continue

    face = result["faces"][0]
    area = face["facial_area"]
    x, y, w, h = (area[key] for key in ("x", "y", "w", "h"))
    cx, cy = x + w / 2, y + h / 2
    half_w, half_h = w * PREVIEW_CROP_SCALE / 2, h * PREVIEW_CROP_SCALE / 2
    box = (int(cx - half_w), int(cy - half_h), int(cx + half_w), int(cy + half_h))
    gender_scores = face.get('gender', {})
    gender_label, gender_score = max(gender_scores.items(), key=lambda item: item[1]) if gender_scores else ('Unknown', 0.0)
    print(f"{user_path.name} | main face | ignored {result['background_faces_ignored']} smaller faces")
    with Image.open(user_path) as photo:
        crop = photo.crop(box)
    glasses_score = preview_pair_score(crop, *GLASSES_PREVIEW_LABELS)
    glasses_label = (glasses_score >= GLASSES_POSITIVE_THRESHOLD
                     and 2 * glasses_score - 1 >= GLASSES_MIN_SCORE_MARGIN)
    hijab_label = False
    if gender_label.lower() == "woman":
        head = crop.crop((int(crop.width * .05), 0, int(crop.width * .95),
                          int(crop.height * .85)))
        hijab_scores = []
        for view in (crop, head):
            for yes, no in HIJAB_PROMPT_PAIRS:
                hijab_scores.append(preview_pair_score(view, yes, no))
        hijab_score = float(np.mean(hijab_scores))
        hijab_label = (hijab_score > .5 and hijab_score >= HIJAB_POSITIVE_THRESHOLD
                       and 2 * hijab_score - 1 >= HIJAB_MIN_SCORE_MARGIN)
    FACE_PREVIEW_RESULTS[user_path.stem] = {"gender": gender_label,
                                    "glasses": bool(glasses_label),
                                    "hijab": bool(hijab_label),
                                    "source": "cell2_preview"}
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(crop)
    ax.set_title(f'{user_path.name}\nDeepFace: {gender_label} ({gender_score:.1f}%)'
                  f'\nGlasses: {"Yes" if glasses_label else "No"} | '
                  f'Hijab: {"Yes" if hijab_label else "No"}')
    ax.axis('off')
    plt.show()
    plt.close(fig)
    shown += 1
print(f"Displayed {shown} main face crops (limit: {PREVIEW_LIMIT}).")


In [ ]:
# ============================================================
# 3. STEP 1 - CHECK IMAGE + ANALYSE FACE
# ============================================================
class Rejected(Exception):
    """Image is blocked; the message is the reason shown to the user."""


def load_image(path):
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
        raise Rejected("Image cannot be read (not an image or corrupted)")
    return img_bgr


def clip_scores(pil_img, labels):
    res = clip(pil_img, candidate_labels=list(labels))
    return {r["label"]: float(r["score"]) for r in res}


# ---------------- scope: block animals, code, cartoons, objects ----------------
SCOPE_LABELS = {
    "person":  "a photo of a real human person showing their face",
    "animal":  "a photo of an animal, a pet, a cat, a dog or a bird",
    "text":    "a screenshot of computer code, text, a document or a website",
    "cartoon": "a cartoon, a drawing, anime, a 3D render or an illustration",
    "object":  "a photo of an object, food, a car, a building or a landscape with no people",
}

def check_scope(pil_img):
    s = clip_scores(pil_img, SCOPE_LABELS.values())
    scores = {k: s[v] for k, v in SCOPE_LABELS.items()}
    top = max(scores, key=scores.get)
    if top != "person" or scores["person"] < SCOPE_MIN_PERSON_SCORE:   # reject only if it looks like something else
        raise Rejected(f"Out of scope: looks like '{top}' (person score {scores['person']:.0%})")
    return scores


# ---------------- face: exactly one main face (DeepFace worker) ----------------
def detect_main_face(user_path):
    res = run_deepface_worker(user_path)
    if res.get("error") == "unreadable":
        raise Rejected("Image cannot be read (not an image or corrupted)")
    if res.get("error") == "multiple_main_faces":
        raise Rejected(f"Multiple main faces ({res['main_faces']}) — upload a photo with one clear foreground person")
    if not res["faces"]:
        raise Rejected("No clear face detected (too small, blurry or covered)")
    return res["faces"][0], res["background_faces_ignored"]


def crop_face(img_bgr, area, scale=CROP_SCALE):
    h_img, w_img = img_bgr.shape[:2]
    x, y, w, h = int(area["x"]), int(area["y"]), int(area["w"]), int(area["h"])
    cx, cy = x + w // 2, y + h // 2
    nw, nh = int(w * scale), int(h * scale)
    x1, y1 = max(0, cx - nw // 2), max(0, cy - nh // 2)
    x2, y2 = min(w_img, cx + nw // 2), min(h_img, cy + nh // 2)
    crop = img_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        raise Rejected("Face crop is empty")
    return crop


# ---------------- gender (DeepFace result from the worker, CLIP fallback) ----------------
def detect_gender(face, crop_bgr, key):
    if "gender" in OVERRIDES.get(key, {}):
        return OVERRIDES[key]["gender"], 100.0, "override"
    conf = 0.0
    if face.get("gender"):
        g, conf = max(face["gender"].items(), key=lambda kv: kv[1])
        g = "Man" if g.lower().startswith("man") else "Woman"
        if conf >= MIN_GENDER_CONFIDENCE:
            return g, conf, "deepface"
    pil = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    s = clip_scores(pil, ["a photo of a man", "a photo of a woman"])
    g2 = "Man" if s["a photo of a man"] >= s["a photo of a woman"] else "Woman"
    c2 = max(s.values())
    if c2 < MIN_CLIP_GENDER_CONFIDENCE:
        raise Rejected(f"Gender uncertain (DeepFace {conf:.0f}%, CLIP {c2:.0%})")
    return g2, c2 * 100, "clip"


# ---------------- glasses / hijab (CLIP, multi-view) ----------------
def create_image_views(image):
    w, h = image.size
    return [
        image,
        ImageEnhance.Contrast(image).enhance(1.10),
        ImageEnhance.Sharpness(image).enhance(1.08),
        image.crop((0, 0, w, max(1, int(h * 0.78)))),                                  # head
        image.crop((int(w*.06), int(h*.06), int(w*.94), int(h*.94))),                   # centre
        image.crop((int(w*.18), int(h*.12), int(w*.82), int(h*.55))),                   # eyes
        image.crop((int(w*.08), int(h*.03), int(w*.92), int(h*.88))),                   # head + neck
    ]

def classify_binary(views, pos_prompts, neg_prompts):
    pos_all, neg_all = [], []
    for v in views:
        for p, n in zip(pos_prompts, neg_prompts):
            s = clip_scores(v, [p, n])
            tot = s[p] + s[n]
            pos_all.append(s[p] / tot if tot else 0)
            neg_all.append(s[n] / tot if tot else 0)
    pos, neg = float(np.mean(pos_all)), float(np.mean(neg_all))
    return {"positive": pos, "negative": neg, "margin": abs(pos - neg)}

def yes_no(scores, thr, margin):
    if scores["positive"] >= thr and scores["positive"] > scores["negative"] and scores["margin"] >= margin:
        return True
    return False   # "No" and "Uncertain" both count as no

def detect_glasses(views, gender):
    who = "man" if gender == "Man" else "woman"
    s = classify_binary(
        views,
        [f"a {who} wearing eyeglasses", f"a {who} wearing glasses", f"a {who} wearing prescription spectacles"],
        [f"a {who} not wearing eyeglasses", f"a {who} without glasses", f"a {who} without prescription spectacles"],
    )
    return yes_no(s, GLASSES_POSITIVE_THRESHOLD, GLASSES_MIN_SCORE_MARGIN), s

def detect_beard(image):
    width, height = image.size
    lower_face = image.crop((0, int(height * .28), width, height))
    scores = classify_binary(
        (image, lower_face, ImageEnhance.Contrast(lower_face).enhance(1.10)),
        ["a man with a visible beard", "a man with clear facial hair",
         "a man with a beard and moustache"],
        ["a clean-shaven man with no beard", "a man with no moustache or stubble",
         "a man with a smooth bare chin and cheeks"],
    )
    return yes_no(scores, BEARD_POSITIVE_THRESHOLD, BEARD_MIN_SCORE_MARGIN), scores


# Keep Step 1 runnable when this cell is rerun in an older notebook kernel.
if "classify_hijab" not in globals():
    HIJAB_PROMPT_PAIRS = [
        ("a photo of a woman wearing a hijab that covers her hair and neck",
         "a photo of a woman with visible uncovered hair around her face"),
        ("a portrait of a woman in a headscarf wrapped around her head",
         "a portrait of a woman with her natural hair visible around her head"),
        ("a woman wearing a cloth scarf that frames her face and covers her hair",
         "a woman with an uncovered hairstyle visible on top and at the sides"),
    ]

    def classify_hijab(image):
        width, height = image.size
        head = image.crop((int(width * .05), 0, int(width * .95), int(height * .85)))
        labels = [label for pair in HIJAB_PROMPT_PAIRS for label in pair]
        view_scores = []
        for view in (image, head):
            scores = clip_scores(view, labels)
            view_scores.extend(scores[yes] / max(scores[yes] + scores[no], 1e-12)
                               for yes, no in HIJAB_PROMPT_PAIRS)
        positive = float(np.mean(view_scores))
        report = {"positive": positive, "negative": 1 - positive,
                  "margin": abs(2 * positive - 1)}
        return positive > .5, report

# ---------------- STEP 1 main ----------------
def analyse_user(user_path):
    user_path = Path(user_path)
    key = user_path.stem
    if user_path.suffix.lower() not in IMAGE_EXTENSIONS:
        raise Rejected(f"Unsupported file type: {user_path.suffix}")

    img_bgr = load_image(user_path)
    face, ignored_background_faces = detect_main_face(user_path)
    area = face["facial_area"]
    crop_bgr = crop_face(img_bgr, area)
    crop_pil = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    scope = check_scope(crop_pil)
    gender, g_conf, g_src = detect_gender(face, crop_bgr, key)

    views = create_image_views(crop_pil)
    preview = globals().get("FACE_PREVIEW_RESULTS", {}).get(key, {})
    if "glasses" in preview:
        glasses = bool(preview["glasses"])
        glasses_s = {"source": preview.get("source", "cell2_preview")}
    else:
        glasses, glasses_s = detect_glasses(views, gender)
    beard, beard_s = detect_beard(crop_pil) if gender == "Man" else (False, None)
    hijab, hijab_s = classify_hijab(crop_pil) if gender == "Woman" else (False, None)

    ov = OVERRIDES.get(key, {})
    if "glasses" in ov: glasses = bool(ov["glasses"])
    if "hijab" in ov and gender == "Woman": hijab = bool(ov["hijab"])
    if "beard" in ov and gender == "Man": beard = bool(ov["beard"])

    tags = [gender] + (["Glasses"] if glasses else []) + (["Hijab"] if hijab else [])
    crop_path = CROP_DIR / f"{key}_{'_'.join(tags)}.jpg"
    cv2.imwrite(str(crop_path), crop_bgr)

    info = {
        "key": key, "user_path": str(user_path), "crop_path": str(crop_path),
        "face_box": [int(area["x"]), int(area["y"]), int(area["w"]), int(area["h"])],
        "faces": 1, "background_faces_ignored": ignored_background_faces,
        "gender": gender, "gender_conf": round(g_conf, 1), "gender_src": g_src,
        "glasses": glasses,
        "glasses_source": glasses_s.get("source", "step1_clip") if isinstance(glasses_s, dict) else "step1_clip",
        "hijab": hijab, "beard": beard,
        "avatar": "Woman_Hijab" if hijab else gender,
    }
    print(f"  Main face: 1 | Smaller faces ignored: {ignored_background_faces} | Gender: {gender} ({g_conf:.1f}% {g_src}) | "
          f"Glasses: {'Yes' if glasses else 'No'} | Hijab: {'Yes' if hijab else 'No'} | "
          f"Beard: {'Yes' if beard else 'No'} | Avatar: {info['avatar']}")
    return info

# Run Step 1 for every user image and display each result.
step1_files = sorted(p for p in USERS_DIR.iterdir()
                     if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS) if USERS_DIR.exists() else []
if not step1_files:
    print("No supported images found in users/.")
for user_path in step1_files:
    preview_info = analyse_user(user_path)
    print(f"Selected avatar: {preview_info['avatar']} | Face crop: {preview_info['crop_path']}")
    with Image.open(preview_info["crop_path"]) as preview_face:
        display(preview_face.copy())
    display(preview_info)


In [ ]:
# ============================================================================
# CELL 8: AVATAR FROM THE CROPPED FACE  (visor version, 40 s time budget)
#   * Takes the Step-1 cropped face and generates the avatar from it. Nothing else.
#   * Similarity validation + best-of-N candidates, stopped by TIME_BUDGET (set VALIDATE = False for ONE plain pass).
#   * The avatar's own visor glasses are ALWAYS used for ALL avatars (Man / Woman / Woman_Hijab) and REPLACE the
#     user's own glasses.
#   * 3rd reference image = close-up of the exact visor of the SELECTED avatar (USE_VISOR_REF).
# Needs cells 1-5: pipe, analyse_user, set_lora, lora_kwargs, Rejected, STEPS, CFG, SEED, LORA_STRENGTH,
#                  AVATAR_DIR, USERS_DIR, OUTPUT_DIR
# ============================================================================
import os, sys, json, time, atexit, subprocess, textwrap, threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import torch
import cv2
import pandas as pd
from PIL import Image, ImageFilter, ImageOps

for _need in ("analyse_user", "set_lora", "lora_kwargs", "pipe", "Rejected", "PROJECT_DIR"):
    assert _need in globals(), f"'{_need}' is missing - run cells 1-5 first."

if "fit16" not in globals():
    def fit16(img):
        W, H = (img.width // 16) * 16, (img.height // 16) * 16
        return img if (W, H) == img.size else img.resize((W, H), Image.LANCZOS)

if "to_black_bg" not in globals():
    def to_black_bg(img, thresh=40):
        a = np.asarray(img.convert("RGB")).astype(np.int16)
        dist = (255 - a).max(axis=2)
        n, lab = cv2.connectedComponents((dist < thresh).astype(np.uint8), connectivity=4)
        border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))) - {0}
        bg = np.isin(lab, list(border)).astype(np.float32)
        bg = cv2.GaussianBlur(cv2.dilate(bg, np.ones((3, 3), np.uint8), 1), (5, 5), 0)
        alpha = np.maximum(bg, cv2.GaussianBlur(bg, (9, 9), 0) * np.clip(1 - dist / (thresh * 3), 0, 1))
        return Image.fromarray((a * (1 - alpha[..., None])).clip(0, 255).astype(np.uint8))

# ------------------------------------------------------------------ SETTINGS
RUN_FOLDER        = True
TEST_AVATARS      = {"Man":         AVATAR_DIR / "Saytara_male_v2.png",
                     "Woman":       AVATAR_DIR / "Saytara_Femal_clean.png",
                     "Woman_Hijab": AVATAR_DIR / "Saytara_hijab.jpg"}
ID_LORA_MULT      = 1.0          # raise to ~1.15 if the face still drifts (stronger likeness, risks copying lighting)
TEST_LORA         = LORA_STRENGTH * ID_LORA_MULT
FALLBACK_SCALE    = 1.15        # only used if the Step-1 crop file is missing (crop from the original photo)
HIJAB_SCALE       = 1.25
REF_SIZE          = 1024
STYLE_MODE        = "comic"     # "semi_real" | "comic" | "off"
KEEP_USER_EXPRESSION = True

# ---- exact visor reference of EACH avatar (3rd image, "Picture 3") ----
USE_VISOR_REF     = True        # False = text description only (2 images)
VISOR_REFS        = {           # box = (x0, y0, x1, y1) as fractions of the template; "image" = optional separate close-up file
    "Man":         {"file": TEST_AVATARS["Man"],         "box": (0.28, 0.22, 0.65, 0.40), "image": None},
    "Woman":       {"file": TEST_AVATARS["Woman"],       "box": (0.25, 0.29, 0.63, 0.49), "image": None},
    "Woman_Hijab": {"file": TEST_AVATARS["Woman_Hijab"], "box": (0.26, 0.25, 0.64, 0.45), "image": None},
}

# ---- similarity validation / optimisation ----
VALIDATE          = True        # score every candidate (face similarity, jaw ratio, visor) and keep the best
BEST_OF_N         = 3           # max candidates per user (1 = single pass); TIME_BUDGET can stop earlier
VISOR_EXTRA_TRIES = 2           # extra candidates if NONE of the first ones has the visor (only if they fit the budget)
TIME_BUDGET       = 40.0        # hard cap in seconds per image (Step 1 + generation + scoring)
ID_LORA_SCHEDULE  = (1.0, 1.1, 1.2)    # put the multiplier that wins most often FIRST
STOP_ID           = 0.55        # stop early when similarity reaches this AND the visor is present
W_JAW             = 0.8         # face-shape (width/height) mismatch costs more (fat stays fat, thin stays thin)
W_NO_VISOR        = 5.0         # a missing visor must lose to any candidate that has it
VERIFY_MODEL      = "ArcFace"
VERIFY_DETECTOR   = "retinaface"
VERIFY_MAX_SIDE   = 512
AVATAR_HEAD_BOX   = {"Man": (0.25, 0.05, 0.68, 0.62), "Woman": (0.25, 0.05, 0.68, 0.62),
                     "Woman_Hijab": (0.30, 0.08, 0.72, 0.72)}   # fraction box of the head inside each avatar

TEST_DIR = OUTPUT_DIR / "prompt_test"
TEST_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------ PROMPTS
_SEMI_REAL = (
    "render the swapped head in the semi-realistic digital painting style of Picture 1: a realistic, true-to-life "
    "face with natural proportions and a recognisable likeness, painted with smooth airbrushed skin and soft "
    "realistic shading (not flat cartoon colors, not a photo and not a vector icon). subtle thin dark line work only "
    "around the eyes, lips, hair strands and jacket edges. detailed realistic eyes with crisp catchlights, soft "
    "glossy lips, natural eyebrows, fine individually painted hair strands with a glossy sheen, warm natural skin "
    "tones with gentle highlights, a strong red rim light along the hair, forehead and cheek edge, and cool blue "
    "fill light on the shadow side, matching the color palette and lighting of the jacket and emblem. the head "
    "blends seamlessly into the painting with no visible swap seam and no harsh photographic texture, "
    "and keeps exactly the face of Picture 2: same face shape, jaw, hairline, facial hair and the real eye color of "
    "Picture 2 (do not turn the eyes blue). "
    "keep the natural age, forehead lines, smile lines, under-eye detail and any grey in the beard or hair of "
    "Picture 2; the airbrushed finish must not slim the face, soften the jaw or change any proportion. "
    "sharp detail, high quality, 4k.")
_COMIC = (
    "render the swapped head fully in the illustration style of Picture 1, not as a photo: a premium digital "
    "comic-book vector portrait with bold clean ink outlines of consistent line weight, cel-shaded skin with smooth "
    "airbrushed gradients and crisp highlight and shadow shapes, rich saturated warm skin tones, individually inked "
    "hair strands with a glossy sheen, glossy lips and sharp eye catchlights, a strong red rim light along the hair, "
    "forehead, cheek and jaw edge and cool blue fill shadows on the opposite side, matching the line quality, color "
    "palette and lighting of the jacket and emblem. the head blends seamlessly into the illustration with no "
    "photographic texture and no swap seam, while still looking exactly like the person in "
    "Picture 2: same face shape, hairline, facial hair and the real eye color of Picture 2 (do not turn the eyes "
    "blue). sharp detail, high quality, 4k.")
STYLE_TEXT = ("the painted finish applies to the rendering only and must never change the face proportions. " +
              {"semi_real": _SEMI_REAL, "comic": _COMIC}[STYLE_MODE]) if STYLE_MODE in ("semi_real", "comic") \
             else "high quality, sharp details, 4k."

_EXPR = ("copy the head rotation and eye direction from Picture 1, but keep the facial expression and smile of Picture 2"
         if KEEP_USER_EXPRESSION else
         "copy the direction of the eye, head rotation, micro expressions from Picture 1")
BFS = ("head_swap: start with Picture 1 as the base image, keeping its lighting, environment, and background. "
       "remove the head from Picture 1 completely and replace it with the head from Picture 2, strictly "
       "preserving the face, hair, eye color and nose structure of Picture 2. " + _EXPR + ". ")

IDENTITY = (
    "IDENTITY IS THE TOP PRIORITY: the result must look like the same real person as Picture 2, not like the person "
    "or the face shape in Picture 1. copy the exact face geometry of Picture 2: overall face width-to-height ratio, "
    "forehead height and width, cheekbone position, full cheeks, jaw width, jawline angle and chin shape (never slim, "
    "narrow, sharpen, shorten or elongate the face, never make it more handsome, younger or more glamorous), the exact "
    "nose length, bridge and nostril width, the exact lip shape and thickness and the same mouth expression (a "
    "closed-mouth smile stays closed, an open smile with visible teeth stays open with the same visible teeth), the "
    "same eyebrow shape, thickness and spacing, the same eye shape, eye size and eye distance, the exact eye color of "
    "Picture 2 (dark brown stays dark brown, never lighten to blue, green or grey), the same skin tone, apparent age, "
    "smile lines and under-eye detail. keep every facial landmark in the same relative position "
    "as Picture 2; only the painting style comes from Picture 1. ")

# body build: heavier stays a little fuller, thinner stays a little thinner
BUILD = (
    "BUILD: match the fullness of the person in Picture 2. if the face is round, full or heavy, keep it a little "
    "fuller: fuller cheeks, softer wider jaw, a fuller chin (even a soft double chin) and a thicker neck. if the face "
    "is thin or lean, keep it a little thinner: leaner cheeks, a defined narrower jaw, visible cheekbones and a "
    "slimmer neck. if average, keep it average. never slim down a full face and never fatten a thin face, and never "
    "copy the build of the person in Picture 1. ")

# clean jawline
FACE_CLEAN = (
    "FACE QUALITY: a clean, continuous, smooth jawline and chin contour from ear to chin on both sides, symmetrical and "
    "undistorted, joining the neck naturally, with no warped, melted, doubled, broken or jagged jaw outline. the skin "
    "is clean, smooth and evenly painted in one natural skin tone. ")

# ---- CRITICAL VISOR REQUIREMENT ----
VISOR_MANDATORY = (
    "CRITICAL REQUIREMENT: The face MUST wear the exact angular electric-blue wraparound visor "
    "from Picture 1 or 3. This is non-negotiable. No ordinary glasses, no sunglasses, no other eyewear. "
    "Only the futuristic visor with the clear transparent lens and blue frame. ")

# ---- THE AVATAR VISOR (used for ALL avatars) ----
VISOR_LOOK_MAN = (
    "the visor is exactly the one worn in Picture 1: a single flat, angular, futuristic wraparound shield made of ONE "
    "continuous transparent panel, with a perfectly straight horizontal top edge running just under the eyebrows from "
    "the outer eye corner on one side, across the bridge of the nose, to the temple on the other side; sharp "
    "chamfered (cut, faceted) lower corners and a shallow notch for the nose; a thin light electric-blue frame line "
    "along its edges; a chunky faceted cyan-blue corner block with a white highlight at the outer end on the side "
    "nearest the camera; and a thick silver-white and blue arm running back along the temple to the ear on the far "
    "side. the lens is clear and transparent with only a very light icy-blue tint and a few crisp white diagonal "
    "glints, so the eyes, eyelashes, eyelids and eyebrows of Picture 2 stay sharp and fully visible through it. ")

VISOR_LOOK_DEFAULT = (
    "the visor is exactly the large angular futuristic shield worn in Picture 1: ONE continuous transparent panel "
    "covering both eyes, with a straight thin electric-blue upper rim just under the eyebrows, broad faceted outer "
    "corners, a shallow V-shaped lower edge around the nose, cyan-blue side blocks and slim arms returning to both "
    "temples. the lens is clear and transparent with only a very light icy-blue tint and a few crisp white glints, "
    "so the eyes, eyelashes, eyelids and eyebrows of Picture 2 stay sharp and fully visible through it. do not turn "
    "it into ordinary eyeglasses or two separate lenses. ")

VISOR_REF_TEXT = (
    "Picture 3 is a close-up of the exact visor glasses: copy this visor exactly onto the face (same shape, straight "
    "top edge, angular corners, blue frame, clear lens, corner blocks and side arms). use ONLY the glasses from "
    "Picture 3, never its face. ")

VISOR_CORE = (
    "EYEWEAR (mandatory): the face wears the Picture 1 visor and NOTHING else. if Picture 2 wears eyeglasses of any "
    "kind (black, thick, round, rectangular, thin metal or clear), DELETE them: erase their frames, rims, nose pads, "
    "arms and lens reflections completely and paint bare natural skin where they were, then put the visor on top. ")

VISOR_NEG = (
    "exactly ONE pair of glasses on the face: no black frame, no thick rims, no second frame, no double lines above "
    "or below the visor, no round or rectangular eyeglasses, no rounded goggles, no sunglasses, no opaque or solid "
    "blue lens. ")

# the visor must be sized and placed for THIS face
VISOR_FIT = (
    "fit the visor to the face of Picture 2: its width spans exactly from temple to temple of that face and is never "
    "wider than the face, it sits level across the bridge of the nose directly over both eyes with the eyes centered "
    "behind the lens, it follows the same head tilt and perspective as the face, and its arms end at the temples and "
    "ears. not oversized, not floating, not tilted, not sliding off the face. ")

VISOR_RETRY = (
    "REMINDER: ordinary or thick eyeglasses must NOT appear anywhere; only the angular blue-framed clear wraparound "
    "visor of Picture 1 appears on the face, clearly visible. ")

HIJAB_VISOR_NOTE = (
    "the visor sits on the face inside the hijab opening: it is no wider than the visible face between the two edges "
    "of the hijab, its arms and corner blocks tuck against the hijab at the temples (they may rest slightly over the "
    "fabric edge), the hijab edge above the eyebrows stays fully visible and is not covered, and the visor never "
    "sticks out beyond the hijab outline or covers the forehead fabric. the hijab fabric is not deformed by it. the "
    "visor MUST be clearly visible on her face. ")

HIJAB_USER = (
    "she wears her own hijab from Picture 2, copied exactly: the same fabric colors (including any two-tone or "
    "lighter under-scarf), the same sheen and soft folds, wrapped the same way tightly around the face with the "
    "edge sitting at the same place on the forehead, cheeks and under the chin, covering all hair, both ears and the "
    "neck, then falling in soft drapes onto the shoulders and chest and flowing into the jacket collar of Picture 1. "
    "no hair visible, no pins, brooches or patterns added, the fabric is not changed to another color. only her head "
    "and hijab are taken from Picture 2; ignore her clothes, cardigan, shirt, body and background. ")

HIJAB_FACE = (
    "keep her natural look from Picture 2: the full natural face with its full cheeks and soft jaw (do not slim or "
    "shrink it), her natural skin tone and smile lines, her natural makeup level and "
    "lip color only - do NOT add eyeliner, eyeshadow, long lashes, contouring or glamour retouching, and do not make "
    "her look younger. the hijab frames the face exactly like Picture 2, not looser and not further back. ")

FEMALE_FACE = (
    "FEMALE FACE DETAILS: copy her face details exactly from Picture 2: the eyebrow shape, thickness and arch, the eye "
    "shape, size and eyelid fold, her natural lash level, the nose width, bridge and tip, the lip shape, fullness and "
    "natural lip color, the smile with the same visible teeth, the cheek fullness, the forehead, the face shape and "
    "chin, her skin tone and dimples. use only a subtle natural makeup level like Picture 2; do not "
    "add glamour makeup, strong lipstick, long lashes or contouring, and do not make her look younger or thinner. "
    "from Picture 1 take only the hairstyle, jacket, painting style and visor. ")

HAIR_USER = (
    "HAIR: ignore the hairstyle of Picture 1 completely and keep the exact hair of Picture 2: the same color (never "
    "add red, orange or blond highlights), length, texture (curly, wavy or straight), volume, hairline, parting and "
    "side length. if Picture 2 has short hair keep it short; if Picture 2 is bald, shaved or has a receding hairline "
    "keep the scalp bald or receding exactly as in Picture 2 and do NOT add, grow or paint any hair. do not smooth, "
    "slick back, comb up, restyle or thicken the hair. the only change allowed on the hair is a thin red rim light. ")

def visor_block(info, attempt=0, ref=False):
    look = VISOR_LOOK_MAN if info.get("avatar") == "Man" else VISOR_LOOK_DEFAULT
    return (VISOR_CORE + (VISOR_REF_TEXT if ref else "") + look + VISOR_NEG + VISOR_FIT +
            (VISOR_RETRY if attempt > 0 else ""))

def build_prompt(info, attempt=0, ref=False):
    V = visor_block(info, attempt, ref)
    closing = ("keep the jacket, emblem and solid black background of Picture 1 unchanged. "
               "final check: the face and build match Picture 2, the jawline is clean, and the angular blue clear "
               "visor of the avatar is on the face with no other glasses.")
    if info.get("hijab", False):
        return (f"{VISOR_MANDATORY}{BFS}{IDENTITY}{BUILD}{V}{HIJAB_VISOR_NOTE}"
                f"{HIJAB_FACE}{HIJAB_USER}{FACE_CLEAN}{STYLE_TEXT} {closing} the visor is present and fits her face.")
    if info["gender"] == "Woman":
        return (f"{VISOR_MANDATORY}{BFS}{IDENTITY}{FEMALE_FACE}{BUILD}"
                "her hair is styled exactly like Picture 1: long black hair with red and orange highlights woven "
                "throughout, styled in a high voluminous bun or updo at the crown, sleek and professionally polished, "
                f"framing the face. {V}{FACE_CLEAN}{STYLE_TEXT} " + closing)
    return (f"{VISOR_MANDATORY}{BFS}{IDENTITY}{BUILD}{HAIR_USER}"
            "FACIAL HAIR: keep the facial hair of Picture 2 exactly as it is: if Picture 2 has a moustache, goatee, "
            "beard or stubble keep the same shape, coverage, length, density and grey or black color with a neat "
            "natural edge; if Picture 2 is clean-shaven keep the skin smooth and add no facial hair. do not copy any "
            f"facial hair from Picture 1. {V}{FACE_CLEAN}{STYLE_TEXT} " + closing)

# ------------------------------------------------------------------ IMAGES
def avatar_on_black(path):
    img = Image.open(path)
    if img.mode in ("RGBA", "LA", "P") or "transparency" in img.info:
        img = img.convert("RGBA")
        img = Image.alpha_composite(Image.new("RGBA", img.size, (0, 0, 0, 255)), img)
    img = img.convert("RGB")
    a = np.asarray(img)
    if np.concatenate([a[:8, :8], a[:8, -8:], a[-8:, :8], a[-8:, -8:]]).mean() > 200:
        img = to_black_bg(img)
    return fit16(img)

_AV = {}
def load_avatar(key):
    if key not in _AV:
        path = TEST_AVATARS.get(key)
        if path is None or not Path(path).exists():
            path = AVATAR_PATHS[key]
        _AV[key] = (avatar_on_black(path), Path(path).name)
    return _AV[key]

_VR = {}
def load_visor_ref(key):
    """Close-up of the exact visor of the SELECTED avatar on a black square (Picture 3). None if disabled / missing."""
    if not USE_VISOR_REF or key not in VISOR_REFS:
        return None
    if key not in _VR:
        cfg, img = VISOR_REFS[key], None
        if cfg.get("image") and Path(cfg["image"]).exists():
            crop = avatar_on_black(cfg["image"])
        elif Path(cfg["file"]).exists():
            src = avatar_on_black(cfg["file"])
            W, H = src.size
            x0, y0, x1, y1 = cfg["box"]
            crop = src.crop((int(x0 * W), int(y0 * H), int(x1 * W), int(y1 * H)))
        else:
            crop = None
            print(f"  (visor reference not found: {cfg['file']} - using the text description only)")
        if crop is not None:
            s = max(crop.size)
            canvas = Image.new("RGB", (s, s), "black")
            canvas.paste(crop, ((s - crop.width) // 2, (s - crop.height) // 2))
            img = canvas.resize((768, 768), Image.LANCZOS)
            img.save(TEST_DIR / f"_visor_ref_{key}.png")        # check this file: it must show the glasses clearly
        _VR[key] = img
    return _VR[key]

def pad_square(img, size=REF_SIZE):
    """Square without stretching; the padding is a blurred copy of the photo."""
    s = max(img.size)
    bg = img.resize((s, s), Image.LANCZOS).filter(ImageFilter.GaussianBlur(max(2, s // 18)))
    bg.paste(img, ((s - img.width) // 2, (s - img.height) // 2))
    return bg.resize((size, size), Image.LANCZOS)

def load_face(info):
    """The Step-1 cropped face (crop/<key>_...jpg). Falls back to a crop of the original via info['face_box']."""
    p = info.get("crop_path")
    if p and Path(p).exists():
        return pad_square(Image.open(p).convert("RGB"))
    img = ImageOps.exif_transpose(Image.open(info["user_path"])).convert("RGB")
    x, y, w, h = info["face_box"]
    s = max(w, h) * (HIJAB_SCALE if info.get("hijab") else FALLBACK_SCALE)
    cx, cy = x + w / 2, y + h / 2 - h * 0.05
    return pad_square(img.crop((int(max(0, cx - s)), int(max(0, cy - s)),
                                int(min(img.width, cx + s)), int(min(img.height, cy + s)))))

def check_visor(out_img):
    """Blue visor pixels must be present on the LEFT lens, the BRIDGE and the RIGHT lens (all avatars). Three separate
    regions keep the blue emblem / a single reflection from counting as a visor."""
    a = np.asarray(out_img.convert("RGB"))
    h, w = a.shape[:2]
    regions = {"left":   a[int(h * .20):int(h * .46), int(w * .26):int(w * .49)],
               "bridge": a[int(h * .22):int(h * .43), int(w * .44):int(w * .57)],
               "right":  a[int(h * .20):int(h * .46), int(w * .52):int(w * .76)]}
    s = {}
    for name, r in regions.items():
        hsv = cv2.cvtColor(np.ascontiguousarray(r), cv2.COLOR_RGB2HSV)
        s[name] = float(((hsv[..., 0] >= 85) & (hsv[..., 0] <= 130) & (hsv[..., 1] > 60) & (hsv[..., 2] > 90)).mean())
    ok = s["left"] > .006 and s["bridge"] > .002 and s["right"] > .006
    return {"visor": "ok" if ok else "weak", "visor_score": round(min(s["left"], s["right"]) + s["bridge"], 4)}

# ------------------------------------------------------------------ SIMILARITY (DeepFace worker, CPU, isolated)
if "close_val_worker" in globals():
    close_val_worker()                                   # kill the worker of a previous run of this cell

VAL_WORKER = PROJECT_DIR / "val_worker.py"
VAL_WORKER.write_text(textwrap.dedent('''
    import os, sys, json, io, contextlib, warnings
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["TF_USE_LEGACY_KERAS"] = "1"
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
    warnings.filterwarnings("ignore")
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        from deepface import DeepFace
    model, detector = sys.argv[1], sys.argv[2]
    print("__READY__", flush=True)
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        req = json.loads(line)
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                reps = DeepFace.represent(img_path=req["img"], model_name=model, detector_backend=detector,
                                          enforce_detection=bool(req["enforce"]), align=True)
            out = {"faces": [{"emb": [float(x) for x in r["embedding"]],
                              "w": int(r["facial_area"].get("w", 0)), "h": int(r["facial_area"].get("h", 0))}
                             for r in reps]}
        except Exception as e:
            out = {"error": type(e).__name__ + ": " + str(e)[:300]}
        print("__JSON__" + json.dumps(out), flush=True)
'''))

VAL_TMP = OUTPUT_DIR / "_val_tmp"
VAL_TMP.mkdir(parents=True, exist_ok=True)
_VW = {"p": None}
_VLOCK = threading.Lock()
_POOL = ThreadPoolExecutor(max_workers=1)

def close_val_worker():
    p = _VW["p"]
    _VW["p"] = None
    if p is not None and p.poll() is None:
        try:
            p.kill()
        except Exception:
            pass
atexit.register(close_val_worker)

def _val_worker():
    p = _VW["p"]
    if p is not None and p.poll() is None:
        return p
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": "-1", "TF_CPP_MIN_LOG_LEVEL": "3", "TF_USE_LEGACY_KERAS": "1"}
    print("  (starting DeepFace worker for similarity - first start ~20-60 s)")
    p = subprocess.Popen([sys.executable, str(VAL_WORKER), VERIFY_MODEL, VERIFY_DETECTOR],
                         stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                         stderr=open(PROJECT_DIR / "val_worker.log", "w"), text=True, bufsize=1, env=env)
    for line in p.stdout:
        if line.startswith("__READY__"):
            _VW["p"] = p
            return p
    raise RuntimeError("DeepFace worker did not start - see val_worker.log")

def _prep(img):
    img = img.convert("RGB")
    k = VERIFY_MAX_SIDE / max(img.size)
    return img.resize((max(1, int(img.width * k)), max(1, int(img.height * k))), Image.LANCZOS) if k < 1 else img

def embed(img, enforce=True, name="out"):
    """Detect + embed the faces of ONE image. -> {"faces": [{emb, w, h}], "width": px} or None."""
    im = _prep(img)
    path = VAL_TMP / f"{name}.png"
    im.save(path)
    with _VLOCK:
        try:
            p = _val_worker()
            p.stdin.write(json.dumps({"img": str(path), "enforce": bool(enforce)}) + "\n")
            p.stdin.flush()
            while True:
                line = p.stdout.readline()
                if not line:
                    close_val_worker()
                    return None
                if line.startswith("__JSON__"):
                    r = json.loads(line[len("__JSON__"):])
                    break
        except BaseException:
            close_val_worker()
            raise
    if "error" in r or not r.get("faces"):
        return None
    return {"faces": r["faces"], "width": im.width}

def embed_user(face):
    return embed(face, True, "user") or embed(face, False, "user")

def _ratio(f, width):
    w, h = f.get("w"), f.get("h")
    return (w / max(h, 1)) if (w and h and w < 0.95 * width) else None

def _cos_dist(a, b):
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    return 1.0 - float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def compare(user, out):
    """Closest face pair, like DeepFace.verify. -> (similarity 0-1, user w/h ratio, avatar w/h ratio)"""
    d, u, o = min(((_cos_dist(u["emb"], o["emb"]), u, o) for u in user["faces"] for o in out["faces"]),
                  key=lambda t: t[0])
    return round(1.0 - d, 4), _ratio(u, user["width"]), _ratio(o, out["width"])

def score_output(user_emb, out, info):
    vis = check_visor(out)
    sim = u_r = o_r = None
    if VALIDATE and user_emb is not None:
        o = embed(out, True, "out")
        if o is None:                                     # fall back to the known head box of the avatar
            box = AVATAR_HEAD_BOX.get(info["avatar"], (0.25, 0.05, 0.68, 0.62))
            W, H = out.size
            o = embed(out.crop((int(box[0] * W), int(box[1] * H), int(box[2] * W), int(box[3] * H))), False, "out")
        if o is not None:
            sim, u_r, o_r = compare(user_emb, o)
    jaw = (abs(o_r - u_r) / u_r) if (u_r and o_r) else None
    score = (sim or 0.0) - W_JAW * (jaw or 0.0) - (W_NO_VISOR if vis["visor"] == "weak" else 0.0)
    return {"id_sim": None if sim is None else round(sim, 3), "jaw_diff": None if jaw is None else round(jaw, 3),
            "score": round(score, 3), **vis}

# ------------------------------------------------------------------ STEP 1 RESULT (reused, not recomputed)
STEP1_INFO = globals().get("STEP1_INFO", {})

def get_info(user_path):
    info = STEP1_INFO.get(str(user_path))
    if info is None or not Path(info["crop_path"]).exists():
        info = STEP1_INFO[str(user_path)] = analyse_user(user_path)
    return info

# ------------------------------------------------------------------ ONE USER
def run_one(user_path, show=True):
    t0 = time.time()                                      # the budget covers EVERYTHING for this image
    user_path = Path(user_path)
    print("=" * 60)
    print("File:", user_path.name)
    rec = {"file": user_path.name}
    try:
        info = get_info(user_path)
        step1_s = time.time() - t0
        avatar, avatar_name = load_avatar(info["avatar"])
        face = load_face(info)
        visor_ref = load_visor_ref(info["avatar"])        # Picture 3 = the visor of THIS avatar (or None)
        images = [avatar, face] + ([visor_ref] if visor_ref is not None else [])
        seed = HIJAB_SEED if info.get("hijab") and "HIJAB_SEED" in globals() else SEED
        W, H = avatar.size
        max_k = BEST_OF_N if VALIDATE else 1
        extras = 0
        fut = _POOL.submit(embed_user, face) if VALIDATE else None    # runs on CPU while the GPU generates try 1
        print(f"  Avatar: {info['avatar']} | seed {seed} | up to {max_k} tries inside {TIME_BUDGET:.0f}s | "
              f"visor ref {'yes' if visor_ref is not None else 'no'} | step1 {step1_s:.1f}s")
        best, k, gen_s, val_s, stopped = None, 0, 0.0, 0.0, "all tries"
        t_loop = time.time()
        while True:
            mult = ID_LORA_SCHEDULE[k % len(ID_LORA_SCHEDULE)]
            set_lora(TEST_LORA * mult)
            prompt = build_prompt(info, attempt=k, ref=visor_ref is not None)   # later tries add a visor reminder
            tg = time.time()
            with torch.no_grad():
                out = pipe(prompt=prompt, image=images, attention_kwargs=lora_kwargs(),
                           width=W, height=H, num_inference_steps=STEPS, guidance_scale=CFG,
                           generator=torch.Generator("cuda").manual_seed(seed + k)).images[0]
            g = time.time() - tg
            tv = time.time()
            user_emb = fut.result() if fut is not None else None
            v = score_output(user_emb, out, info)
            s = time.time() - tv
            gen_s, val_s = gen_s + g, val_s + s
            print(f"  try {k + 1} seed {seed + k} lora x{mult} -> similarity={v['id_sim']} jaw_diff={v['jaw_diff']} "
                  f"visor={v['visor']} score={v['score']} | gen {g:.1f}s score {s:.1f}s")
            if best is None or (v["visor"] == "ok", v["score"]) > (best["v"]["visor"] == "ok", best["v"]["score"]):
                best = {"out": out, "v": v, "k": k + 1, "mult": mult}
            k += 1                                        # k = number of tries done
            if VALIDATE and (v["id_sim"] or 0) >= STOP_ID and v["visor"] == "ok":
                stopped = "good"
                break
            if k >= max_k:
                if best["v"]["visor"] != "ok" and extras < VISOR_EXTRA_TRIES:
                    extras += 1
                    max_k += 1
                    print("  no candidate has the visor yet -> extra try (if it fits the budget)")
                else:
                    break
            per_try = (time.time() - t_loop) / k          # measured cost of one try for THIS user
            if time.time() - t0 + per_try > TIME_BUDGET:
                stopped = "budget"
                print(f"  another try (~{per_try:.0f}s) would pass {TIME_BUDGET:.0f}s -> keeping the best so far")
                break
        out, v = best["out"], best["v"]
        out_path = TEST_DIR / f"{info['key']}_{info['avatar']}.png"
        out.save(out_path)
        rec.update(status="done", gender=info["gender"], hijab=info.get("hijab"), avatar=info["avatar"],
                   output=str(out_path), seconds=round(time.time() - t0, 1), step1_s=round(step1_s, 1),
                   gen_s=round(gen_s, 1), val_s=round(val_s, 1), tries=k, best_try=best["k"], stopped=stopped,
                   lora_mult=best["mult"], id_sim=v["id_sim"], jaw_diff=v["jaw_diff"], score=v["score"],
                   visor=v["visor"], visor_score=v["visor_score"])
        print(f"  BEST: try {best['k']} of {k} | similarity={v['id_sim']} jaw_diff={v['jaw_diff']} visor={v['visor']} "
              f"-> {out_path.name} ({rec['seconds']}s, stopped: {stopped})")
        if v["visor"] != "ok":
            print("  WARNING: the avatar visor is still missing in the best result - check _visor_ref_"
                  f"{info['avatar']}.png and VISOR_REFS['{info['avatar']}']['box'].")
        if show:
            # Display: [avatar] | [user face] | [result]
            u = face.resize((int(face.width * H / face.height), H))
            trio = Image.new("RGB", (avatar.width + u.width + out.width, H), "black")
            trio.paste(avatar, (0, 0))
            trio.paste(u, (avatar.width, 0))
            trio.paste(out, (avatar.width + u.width, 0))
            display(trio.resize((trio.width // 2, trio.height // 2)))
    except Rejected as r:
        rec.update(status="rejected", reason=str(r))
        print("  BLOCKED:", r)
    except Exception as e:
        import traceback
        rec.update(status="error", reason=f"{type(e).__name__}: {e}")
        print("  ERROR:", rec["reason"])
        print("".join(traceback.format_exc().splitlines(True)[-8:]))
    finally:
        set_lora(LORA_STRENGTH)
    return rec

# ------------------------------------------------------------------ WHOLE users/ FOLDER
files = sorted(p for p in USERS_DIR.iterdir() if p.is_file()) if RUN_FOLDER else []
print(f"Found {len(files)} files\n")
if VALIDATE and files:
    _val_worker()                                         # start the worker BEFORE the first image is timed
records = [run_one(p) for p in files]
df = pd.DataFrame(records)
if len(df):
    display(df[[c for c in ["file", "status", "avatar", "id_sim", "jaw_diff", "score", "tries", "best_try",
                 "stopped", "lora_mult", "visor", "seconds", "step1_s", "gen_s", "val_s", "reason"] if c in df]])
    print(df["status"].value_counts().to_string())
    if "seconds" in df:
        print(f"\nseconds per image: mean {df['seconds'].mean():.1f} | max {df['seconds'].max():.1f}")
        print(df["stopped"].value_counts().to_string())